# Trayectoria IA — V3.1 | migración real a scikit-learn 1.9.1

**Entrenamiento reproducible + inferencia para CSV con ID seudónimo + ficha individual.**

Este cuaderno **reentrena** Random Forest y KMeans usando `scikit-learn==1.9.1` e `imbalanced-learn==0.14.2`, guarda el nuevo `.pkl` y su manifest, verifica la plantilla española y crea un ZIP listo para entregar al equipo web.

**Instrucciones de Colab:**
1. Ejecuta la celda de instalación, **reinicia la sesión de ejecución** desde *Entorno de ejecución → Reiniciar sesión* y después ejecuta desde la celda de importaciones hasta el final. (Si acabas de abrir una sesión limpia, normalmente bastará con ejecutar en orden, pero el reinicio evita que queden módulos antiguos importados.)
2. No tienes que subir `data.csv`: si no está presente, se descarga UCI 697. El cuaderno reconstruye un CSV de demostración con 4.424 IDs ficticios, sin nombres.
3. Al final se genera **`Trayectoria_V3_1_LISTO_PARA_WEB_sklearn_1_9_1.zip`** con un `.pkl` **realmente entrenado** en 1.9.1 y resultados de verificación.

**Nota:** el CSV original tiene 36 variables para el modelo. La página recibe 37 columnas: `ID estudiante` + 36 variables; el ID **no** se usa para predecir. Se conservan los nombres y la función de integración `procesar_dataframe()`.


## 0. Instalación de dependencias (Python ≥ 3.11)

La instalación debe efectuarse antes de importar scikit-learn. Si la sesión tenía importada otra versión, **reiníciala** antes de entrenar. `imbalanced-learn==0.14.2` incorpora compatibilidad específica con scikit-learn 1.9. `narwhals` es una dependencia de scikit-learn 1.9.


In [ ]:
%pip -q install "scikit-learn==1.9.1" "imbalanced-learn==0.14.2" "numpy==2.3.5" "scipy==1.17.0" "pandas==2.2.3" "joblib==1.5.3" "threadpoolctl==3.6.0" "narwhals==2.21.2" "ucimlrepo>=0.0.7,<1.0" "openpyxl==3.1.5"

In [ ]:
import json
import hashlib
import platform
import importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn, scipy, joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, accuracy_score, precision_score, recall_score, silhouette_score
from sklearn.cluster import KMeans
from scipy.optimize import linear_sum_assignment
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError('Se necesita Python >= 3.11 para scikit-learn 1.9.1.')
if sklearn.__version__ != "1.9.1":
    raise RuntimeError(f"Se necesita scikit-learn 1.9.1; actual: {sklearn.__version__}. Reinicia el entorno después de instalar las dependencias.")
if importlib.metadata.version('imbalanced-learn') != '0.14.2':
    raise RuntimeError('Se necesita imbalanced-learn 0.14.2 (compatible con scikit-learn 1.9).')
import narwhals
assert importlib.metadata.version('narwhals') == '2.21.2'
print('Dependencias verificadas: scikit-learn 1.9.1, imbalanced-learn 0.14.2, narwhals 2.21.2')
print("Python:", platform.python_version(), "scikit-learn:", sklearn.__version__)


## 1. Dataset de ENTRENAMIENTO (sin IDs)

Se utiliza `data.csv` local si existe; en caso contrario se descarga el dataset original portugués desde UCI ID 697. Nunca se entrena con `ID estudiante`. Los datos de entrenamiento se usan solo aquí: el módulo para la web no necesita descargarlos.


In [ ]:
RUTAS_UCI = [Path("data.csv"), Path("/content/sample_data/data.csv"), Path("../data.csv")]
ruta_local = next((p for p in RUTAS_UCI if p.exists()), None)
if ruta_local:
    origen = pd.read_csv(ruta_local, sep=";")
    origen.columns = [c.strip().replace("Marital status", "Marital Status") for c in origen.columns]
    X = origen.drop(columns=["Target"]).copy()
    target_original = origen["Target"].copy()
    print(f"Datos cargados desde archivo local: {ruta_local}")
else:
    from ucimlrepo import fetch_ucirepo
    dataset = fetch_ucirepo(id=697)
    X = dataset.data.features.copy()
    target_original = dataset.data.targets.squeeze().copy()
    X.columns = [c.strip().replace("Marital status", "Marital Status") for c in X.columns]
    print("Datos descargados desde UCI, ID 697")

assert X.shape[1] == 36, f"Se esperaban 36 características, llegaron {X.shape[1]}"
y = (target_original == "Dropout").astype(int)
FEATURES_MODELO = X.columns.tolist()
print("Estudiantes:", X.shape[0], "Variables:", X.shape[1])
print("Target original:", target_original.value_counts().to_dict())


## 2. Entrenamiento, evaluación e identificación de estudiantes con mayor riesgo

Se replica la estrategia V2 para no cambiar innecesariamente el resultado de la página. SMOTE se aplica solo al entrenamiento (nunca al test). La clasificación binaria es `Dropout` frente a `Enrolled/Graduate`.

**Observación metodológica:** SMOTE sobre códigos categóricos numéricos puede interpolar categorías artificiales. Es una limitación heredada de V2 a revisar antes de usar el sistema para decisiones de alto impacto.

**Control de migración:** se mantiene la configuración del modelo del equipo para evitar modificaciones metodológicas ajenas a la actualización. Se comprueba que el rendimiento sea razonablemente similar al de V3.1, pero cambiar versiones puede alterar valores y por eso se vuelven a calcular todas las predicciones.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# La regresión logística sí necesita escalado.
scaler_log = StandardScaler()
X_train_bal_scaled = scaler_log.fit_transform(X_train_bal)
X_test_scaled = scaler_log.transform(X_test)

print("Train original:", y_train.value_counts().to_dict())
print("Train con SMOTE:", pd.Series(y_train_bal).value_counts().to_dict())
print("Test:", y_test.value_counts().to_dict())


In [ ]:
modelos = {
    "Regresión Logística": LogisticRegression(
        max_iter=3000, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight=None
    )
}

modelos["Regresión Logística"].fit(X_train_bal_scaled, y_train_bal)
modelos["Random Forest"].fit(X_train_bal, y_train_bal)

resultados = {}

for nombre, modelo in modelos.items():
    X_eval = X_test_scaled if nombre == "Regresión Logística" else X_test
    pred = modelo.predict(X_eval)
    proba = modelo.predict_proba(X_eval)[:, 1]

    resultados[nombre] = {
        "F1_desercion": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, proba),
        "Accuracy": accuracy_score(y_test, pred),
        "Precision_desercion": precision_score(y_test, pred),
        "Recall_desercion": recall_score(y_test, pred)
    }

    print(f"\n=== {nombre} ===")
    print(classification_report(y_test, pred, target_names=["No deserta", "Deserta"]))

comparacion = pd.DataFrame(resultados).T.sort_values("F1_desercion", ascending=False)
comparacion


# Barrera mínima de regresión respecto a la evaluación histórica de V3.1.
# No sustituye una evaluación externa en universidades chilenas.
rf_eval = resultados["Random Forest"]
if rf_eval["F1_desercion"] < 0.75 or rf_eval["ROC_AUC"] < 0.89:
    raise RuntimeError(f"La migración degradó demasiado el rendimiento de prueba: {rf_eval}")
print("[OK] Evaluación mínima de regresión aprobada.")

In [ ]:
modelo_riesgo = modelos["Random Forest"]

proba_full = modelo_riesgo.predict_proba(X)[:, 1]

# Umbral operativo inicial. Puede calibrarse posteriormente según el objetivo institucional.
UMBRAL_RIESGO = 0.50

base_perfiles = X.copy()
base_perfiles["probabilidad_desercion"] = proba_full
base_perfiles = base_perfiles[
    base_perfiles["probabilidad_desercion"] >= UMBRAL_RIESGO
].copy()

print(f"Estudiantes para construir perfiles: {len(base_perfiles)} / {len(X)}")


In [ ]:
COLS_CLUSTER = [
    "Age at enrollment",
    "Scholarship holder",
    "Debtor",
    "Tuition fees up to date",
    "Curricular units 1st sem (approved)",
    "Curricular units 1st sem (grade)",
    "Curricular units 2nd sem (approved)",
    "Curricular units 2nd sem (grade)",
    "Displaced",
    "Unemployment rate",
    "Mother's qualification",
    "Father's qualification"
]

scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(base_perfiles[COLS_CLUSTER])

# Comprobación de k. Se conserva k=4 para mantener la lógica del proyecto,
# pero mostramos silhouette para dejar documentada la decisión.
silhouettes = {}
for k in range(2, 8):
    km_tmp = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=20)
    labels_tmp = km_tmp.fit_predict(X_cluster_scaled)
    silhouettes[k] = silhouette_score(X_cluster_scaled, labels_tmp)

print("Silhouette por k:", {k: round(v, 4) for k, v in silhouettes.items()})

K_OPTIMO = 4
kmeans_perfiles = KMeans(
    n_clusters=K_OPTIMO,
    random_state=RANDOM_STATE,
    n_init=20
)
base_perfiles["cluster"] = kmeans_perfiles.fit_predict(X_cluster_scaled)
base_perfiles["cluster"].value_counts().sort_index()


## 3. Caracterización de clusters reales

La tabla muestra **medias del grupo**, no diagnósticos individuales. Los nombres de las variables se traducen para facilitar la lectura, aunque internamente el modelo conserve las columnas UCI en inglés. No interpretes la media de códigos de escolaridad de los padres como un nivel educativo real (son categorías).

In [ ]:
NOMBRES_CLUSTER_ESP = {
    "Age at enrollment": "Edad al matricularse",
    "Scholarship holder": "Beneficiario/a de beca",
    "Debtor": "Deudor/a",
    "Tuition fees up to date": "Aranceles al día",
    "Curricular units 1st sem (approved)": "Asignaturas aprobadas 1S",
    "Curricular units 1st sem (grade)": "Promedio 1S",
    "Curricular units 2nd sem (approved)": "Asignaturas aprobadas 2S",
    "Curricular units 2nd sem (grade)": "Promedio 2S",
    "Displaced": "Desplazado/a",
    "Unemployment rate": "Tasa de desempleo",
    "Mother's qualification": "Nivel educativo de la madre (código UCI)",
    "Father's qualification": "Nivel educativo del padre (código UCI)"
}
perfil_clusters = base_perfiles.groupby("cluster")[COLS_CLUSTER].mean().round(2)
promedio_riesgo = base_perfiles[COLS_CLUSTER].mean().round(2)
display(perfil_clusters.rename(columns=NOMBRES_CLUSTER_ESP))
print("Promedio de estudiantes identificados en riesgo:")
display(promedio_riesgo.rename(index=NOMBRES_CLUSTER_ESP).to_frame("promedio"))
print("Estudiantes por cluster:")
print(base_perfiles["cluster"].value_counts().sort_index().to_dict())


## 4. Perfiles descriptivos, asociados a los cuatro clusters originales

Se preservan los mismos IDs numéricos y el mismo algoritmo V3. Los nombres reflejan las medias verificadas y **no atribuyen causas individuales**:
- Grupo 0: progresión conservada y mayor proporción de desplazados (aprox. 61%; no todos lo son).
- Grupo 1: progresión académica crítica en alumnos con mayor edad de ingreso.
- Grupo 2: progresión académica crítica en alumnos jóvenes.
- Grupo 3: progresión conservada en alumnos con mayor edad de ingreso.

No denominamos un cluster «riesgo económico» porque la deuda no distingue claramente los grupos; esa información aparecerá en **alertas individuales**, sin asumir causalidad. Las etiquetas son descripciones del conjunto portugués y pueden no transferirse automáticamente a Chile.

In [ ]:
PERFILES_SEMANTICOS = {
    "joven_favorable": {
        "perfil": "Progresión conservada y mayor frecuencia de desplazamiento",
        "factores_perfil": "Grupo con aprobación relativamente conservada (aprox. 4 asignaturas en segundo semestre), edad media cercana a 22 años y mayor proporción de estudiantes desplazados (aprox. 61%). La etiqueta describe el grupo; no todos sus integrantes son desplazados.",
        "intervencion_sugerida": "Seguimiento preventivo y revisión individual de condiciones académicas, económicas o logísticas; ofrecer apoyo solo donde exista necesidad identificada."
    },
    "adulto_bajo": {
        "perfil": "Progresión académica crítica en estudiantes de mayor edad",
        "factores_perfil": "Grupo con edad de ingreso media cercana a 31 años y aprobación casi nula en ambos semestres (aprox. 0,06 asignaturas aprobadas en segundo semestre).",
        "intervencion_sugerida": "Priorizar revisión de progresión curricular y acompañamiento académico; explorar barreras individuales sin asumir que la edad sea la causa."
    },
    "joven_bajo": {
        "perfil": "Progresión académica crítica en estudiantes jóvenes",
        "factores_perfil": "Grupo con edad de ingreso media cercana a 23 años y aprobación casi nula, particularmente en segundo semestre (aprox. 0,04 asignaturas aprobadas).",
        "intervencion_sugerida": "Priorizar detección de dificultades académicas, tutorías y seguimiento del avance en los primeros semestres."
    },
    "adulto_favorable": {
        "perfil": "Progresión conservada en estudiantes de mayor edad",
        "factores_perfil": "Grupo con edad media cercana a 28 años y aprobación relativamente conservada (aprox. 3,74 asignaturas aprobadas en segundo semestre).",
        "intervencion_sugerida": "Mantener seguimiento preventivo y consultar si existen barreras no académicas que requieran apoyo."
    }
}
CENTROIDES_REFERENCIA = np.array([
    [22.09, 4.02, 12.13], # joven_favorable
    [30.82, 0.06, 0.48],  # adulto_bajo
    [22.50, 0.04, 0.33],  # joven_bajo
    [28.30, 3.74, 11.84]  # adulto_favorable
])
medias = base_perfiles.groupby("cluster")[[
    "Age at enrollment", "Curricular units 2nd sem (approved)", "Curricular units 2nd sem (grade)"
]].mean()
diferencias = ((medias.to_numpy()[:, None, :] - CENTROIDES_REFERENCIA[None, :, :]) /
               np.array([6.0, 2.0, 5.0])) ** 2
filas, columnas = linear_sum_assignment(diferencias.sum(axis=-1))
if max(diferencias.sum(axis=-1)[filas, columnas]) >= 5:
    raise ValueError("Los clusters cambiaron demasiado; revisar interpretación manualmente antes de exportar.")
CLAVES_PERFILES = ["joven_favorable", "adulto_bajo", "joven_bajo", "adulto_favorable"]
CONFIG_PERFILES = {
    int(f): PERFILES_SEMANTICOS[CLAVES_PERFILES[int(c)]]
    for f, c in zip(filas, columnas)
}
print({k: v["perfil"] for k, v in sorted(CONFIG_PERFILES.items())})


## 5. Conversión de la plantilla española a códigos UCI

Estas tablas son **las mismas** que usa la plantilla compartida con el equipo. El `.pkl` guarda los mapas, así que el servidor no necesita el CSV portugués.

In [ ]:
COLUMNAS_PAGINA_A_UCI = {
    "Estado civil":"Marital Status",
    "Modo de postulación":"Application mode",
    "Orden de postulación":"Application order",
    "Carrera":"Course",
    "Jornada de asistencia":"Daytime/evening attendance",
    "Formación previa":"Previous qualification",
    "Promedio de formación previa":"Previous qualification (grade)",
    "Nacionalidad":"Nacionality",
    "Nivel educativo de la madre":"Mother's qualification",
    "Nivel educativo del padre":"Father's qualification",
    "Ocupación de la madre":"Mother's occupation",
    "Ocupación del padre":"Father's occupation",
    "Puntaje de admisión":"Admission grade",
    "Desplazado/a":"Displaced",
    "Necesidades educativas especiales":"Educational special needs",
    "Deudor/a":"Debtor",
    "Aranceles al día":"Tuition fees up to date",
    "Género":"Gender",
    "Beneficiario/a de beca":"Scholarship holder",
    "Edad al momento de la matrícula":"Age at enrollment",
    "Estudiante internacional":"International",
    "Asignaturas del 1.er semestre (convalidadas)":"Curricular units 1st sem (credited)",
    "Asignaturas del 1.er semestre (inscritas)":"Curricular units 1st sem (enrolled)",
    "Asignaturas del 1.er semestre (evaluaciones)":"Curricular units 1st sem (evaluations)",
    "Asignaturas del 1.er semestre (aprobadas)":"Curricular units 1st sem (approved)",
    "Promedio del 1.er semestre":"Curricular units 1st sem (grade)",
    "Asignaturas del 1.er semestre (sin evaluaciones)":"Curricular units 1st sem (without evaluations)",
    "Asignaturas del 2.º semestre (convalidadas)":"Curricular units 2nd sem (credited)",
    "Asignaturas del 2.º semestre (inscritas)":"Curricular units 2nd sem (enrolled)",
    "Asignaturas del 2.º semestre (evaluaciones)":"Curricular units 2nd sem (evaluations)",
    "Asignaturas del 2.º semestre (aprobadas)":"Curricular units 2nd sem (approved)",
    "Promedio del 2.º semestre":"Curricular units 2nd sem (grade)",
    "Asignaturas del 2.º semestre (sin evaluaciones)":"Curricular units 2nd sem (without evaluations)",
    "Tasa de desempleo":"Unemployment rate",
    "Tasa de inflación":"Inflation rate",
    "PIB":"GDP"
}

MAPAS_TEXTO_A_CODIGO = {
    "Estado civil": {
        "Soltero/a":1, "Casado/a":2, "Viudo/a":3, "Divorciado/a":4,
        "Unión de hecho":5, "Separado/a legalmente":6
    },
    "Modo de postulación": {
        "1.ª fase - contingente general":1, "Ordenanza N.º 612/93":2,
        "1.ª fase - contingente especial (Azores)":5, "Titular de otro curso superior":7,
        "Ordenanza N.º 854-B/99":10, "Estudiante internacional (licenciatura)":15,
        "1.ª fase - contingente especial (Madeira)":16, "2.ª fase - contingente general":17,
        "3.ª fase - contingente general":18, "Ordenanza N.º 533-A/99, b2 (plan diferente)":26,
        "Ordenanza N.º 533-A/99, b3 (otra institución)":27, "Mayor de 23 años":39,
        "Transferencia":42, "Cambio de carrera":43,
        "Titular de diploma de especialización tecnológica":44,
        "Cambio de institución/carrera":51, "Titular de diploma de ciclo corto":53,
        "Cambio de institución/carrera (internacional)":57
    },
    "Carrera": {
        "Tecnologías de Producción de Biocombustibles":33,
        "Diseño de Animación y Multimedia":171, "Servicio Social (vespertino)":8014,
        "Agronomía":9003, "Diseño de Comunicación":9070, "Enfermería Veterinaria":9085,
        "Ingeniería Informática":9119, "Equinocultura":9130, "Gestión":9147,
        "Servicio Social":9238, "Turismo":9254, "Enfermería":9500,
        "Higiene Oral":9556, "Gestión de Publicidad y Marketing":9670,
        "Periodismo y Comunicación":9773, "Educación Básica":9853,
        "Gestión (vespertino)":9991
    },
    "Jornada de asistencia":{"Vespertina":0, "Diurna":1},
    "Género":{"Femenino":0, "Masculino":1},
}

SI_NO = {"No":0, "Sí":1}
for c in [
    "Desplazado/a", "Necesidades educativas especiales", "Deudor/a",
    "Aranceles al día", "Beneficiario/a de beca", "Estudiante internacional"
]:
    MAPAS_TEXTO_A_CODIGO[c] = SI_NO

# Las categorías de educación, nacionalidad y ocupación se definen
# explícitamente en la siguiente celda y se guardarán en el .pkl.


In [ ]:
MAPAS_TEXTO_A_CODIGO["Formación previa"] = {
"Educación secundaria":1,"Educación superior - licenciatura/bachelor":2,"Educación superior - grado":3,
"Educación superior - máster":4,"Educación superior - doctorado":5,"Frecuencia de educación superior":6,
"12.º año no completado":9,"11.º año no completado":10,"Otro - 11.º año":12,"10.º año":14,
"10.º año no completado":15,"Educación básica 3.er ciclo (9.º/10.º/11.º) o equivalente":19,
"Educación básica 2.º ciclo (6.º/7.º/8.º) o equivalente":38,"Curso de especialización tecnológica":39,
"Educación superior - grado (1.er ciclo)":40,"Curso técnico superior profesional":42,
"Educación superior - máster (2.º ciclo)":43}

MAPAS_TEXTO_A_CODIGO["Nacionalidad"] = {
"Portuguesa":1,"Alemana":2,"Española":6,"Italiana":11,"Neerlandesa":13,"Inglesa":14,"Lituana":17,
"Angoleña":21,"Caboverdiana":22,"Guineana":24,"Mozambiqueña":25,"Santomense":26,"Turca":32,
"Brasileña":41,"Rumana":62,"Moldava":100,"Mexicana":101,"Ucraniana":103,"Rusa":105,
"Cubana":108,"Colombiana":109}

QUAL = {
"Educación secundaria - 12.º año o equivalente":1,"Educación superior - licenciatura/bachelor":2,
"Educación superior - grado":3,"Educación superior - máster":4,"Educación superior - doctorado":5,
"Frecuencia de educación superior":6,"12.º año no completado":9,"11.º año no completado":10,
"7.º año (antiguo)":11,"Otro - 11.º año":12,"2.º año complementario de secundaria":13,"10.º año":14,
"Curso general de comercio":18,"Educación básica 3.er ciclo (9.º/10.º/11.º) o equivalente":19,
"Curso complementario de secundaria":20,"Curso técnico-profesional":22,
"Curso complementario de secundaria no concluido":25,"7.º año de escolaridad":26,
"2.º ciclo del curso general de secundaria":27,"9.º año no completado":29,"8.º año de escolaridad":30,
"Curso general de Administración y Comercio":31,"Contabilidad y Administración suplementaria":33,
"Desconocido":34,"No sabe leer ni escribir":35,"Sabe leer sin haber cursado 4.º año":36,
"Educación básica 1.er ciclo (4.º/5.º) o equivalente":37,
"Educación básica 2.º ciclo (6.º/7.º/8.º) o equivalente":38,"Curso de especialización tecnológica":39,
"Educación superior - grado (1.er ciclo)":40,"Curso de estudios superiores especializados":41,
"Curso técnico superior profesional":42,"Educación superior - máster (2.º ciclo)":43,
"Educación superior - doctorado (3.er ciclo)":44}
MAPAS_TEXTO_A_CODIGO["Nivel educativo de la madre"] = QUAL
MAPAS_TEXTO_A_CODIGO["Nivel educativo del padre"] = QUAL

OCC = {
"Estudiante":0,"Representantes/dirigentes y directivos":1,
"Especialistas en actividades intelectuales y científicas":2,
"Técnicos y profesionales de nivel intermedio":3,"Personal administrativo":4,
"Trabajadores de servicios personales, seguridad y vendedores":5,
"Agricultores y trabajadores cualificados agropecuarios, pesca y forestal":6,
"Trabajadores cualificados de industria, construcción y artesanía":7,
"Operadores de instalaciones/máquinas y montaje":8,"Trabajadores no cualificados":9,
"Profesiones de las Fuerzas Armadas":10,"Otra situación":90,"Sin especificar":99,
"Oficiales de las Fuerzas Armadas":101,"Sargentos de las Fuerzas Armadas":102,
"Otro personal de las Fuerzas Armadas":103,"Directores de servicios administrativos y comerciales":112,
"Directores de hotelería, restauración, comercio y otros servicios":114,
"Especialistas en ciencias físicas, matemáticas, ingeniería y afines":121,
"Profesionales de la salud":122,"Docentes":123,
"Especialistas en finanzas, contabilidad, organización administrativa y relaciones públicas/comerciales":124,
"Especialistas TIC":125,"Técnicos intermedios de ciencia e ingeniería":131,
"Técnicos y profesionales intermedios de salud":132,
"Técnicos intermedios de servicios jurídicos, sociales, deportivos y culturales":134,
"Técnicos de tecnologías de información y comunicación":135,
"Oficinistas, secretarios y operadores de procesamiento de datos":141,
"Operadores de datos, contabilidad, estadística, finanzas y registros":143,
"Otro personal de apoyo administrativo":144,"Trabajadores de servicios personales":151,"Vendedores":152,
"Trabajadores de cuidados personales y afines":153,"Personal de protección y seguridad":154,
"Agricultores y productores cualificados orientados al mercado":161,
"Agricultores, ganaderos, pescadores, cazadores y recolectores de subsistencia":163,
"Trabajadores cualificados de construcción, excepto electricistas":171,
"Trabajadores cualificados de metalurgia y afines":172,
"Trabajadores cualificados de impresión, precisión, joyería y artesanía":173,
"Trabajadores cualificados de electricidad y electrónica":174,
"Trabajadores de procesamiento de alimentos, madera, confección y otras industrias/artesanías":175,
"Operadores de instalaciones y máquinas fijas":181,"Trabajadores de montaje":182,
"Conductores de vehículos y operadores de equipos móviles":183,"Trabajadores de limpieza":191,
"Trabajadores no cualificados agropecuarios, pesca y forestal":192,
"Trabajadores no cualificados de industria extractiva, construcción, manufactura y transporte":193,
"Ayudantes de preparación de comidas":194,
"Vendedores ambulantes y prestadores de servicios callejeros":195}
MAPAS_TEXTO_A_CODIGO["Ocupación de la madre"] = OCC
MAPAS_TEXTO_A_CODIGO["Ocupación del padre"] = OCC


## 6. Módulo Python para la web

La siguiente celda escribe `trayectoria_inferencia.py`. El backend seguirá utilizando `procesar_dataframe()` y `obtener_ficha_estudiante()`. El cargador **rechaza** modelos que no sean 1.9.1 y exige manifest válido (con verificación de SHA-256).


In [ ]:
%%writefile trayectoria_inferencia.py
"""Trayectoria V3.1 — inferencia actualizada para scikit-learn 1.9.1.

No entrena modelos ni descarga datos. Solo se debe cargar un .pkl del equipo.
Requisitos y documentación: requirements_inferencia.txt y LEEME_Trayectoria_V3_1.md.
"""
from functools import lru_cache
import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

ID_COLUMNA = 'ID estudiante'
ARTEFACTO_POR_DEFECTO = Path(__file__).resolve().parent / 'trayectoria_pipeline_v3_1.pkl'
VERSION_REQUERIDA = '1.9.1'
COLUMNAS_RESULTADO_ANTERIORES = (
    'probabilidad_desercion', 'porcentaje_riesgo', 'nivel_riesgo', 'en_riesgo',
    'cluster', 'perfil', 'factores_perfil', 'intervencion_sugerida'
)
COLUMNAS_RESULTADO_NUEVAS = ('alertas_individuales', 'recomendaciones_individuales')


@lru_cache(maxsize=4)
def cargar_artefacto(ruta_artefacto=None):
    """Reutiliza un modelo confiable y comprueba versión antes de cargar pickle."""
    ruta = Path(ruta_artefacto).resolve() if ruta_artefacto else ARTEFACTO_POR_DEFECTO
    if not ruta.is_file():
        raise FileNotFoundError(f'No se encontró el artefacto de Trayectoria: {ruta}')
    if sklearn.__version__ != VERSION_REQUERIDA:
        raise RuntimeError(
            f'Este módulo necesita scikit-learn {VERSION_REQUERIDA}; versión actual: '
            f'{sklearn.__version__}. Usa requirements_inferencia.txt.'
        )
    manifiesto = ruta.with_suffix('.manifest.json')
    if not manifiesto.is_file():
        raise FileNotFoundError(
            f'Falta el manifest de compatibilidad: {manifiesto}. '
            'No se acepta un .pkl antiguo sin metadatos.'
        )
    info = json.loads(manifiesto.read_text(encoding='utf-8'))
    version = info.get('scikit_learn')
    if version != VERSION_REQUERIDA:
        raise RuntimeError(
            f'El modelo fue guardado para scikit-learn {version!r} y se '
            f'requiere {VERSION_REQUERIDA}. Vuelve a entrenarlo; no basta con editar el manifest.'
        )
    esperado = info.get('archivo_sha256')
    if esperado:
        h = hashlib.sha256()
        with ruta.open('rb') as f:
            for bloque in iter(lambda: f.read(1024 * 1024), b''):
                h.update(bloque)
        if h.hexdigest() != esperado:
            raise RuntimeError('El .pkl no corresponde con su manifest (SHA-256 diferente).')
    # Nunca aceptar rutas de archivo desde parámetros proporcionados por visitantes.
    artefacto = joblib.load(ruta)
    version = artefacto.get('metadatos', {}).get('scikit_learn')
    if version and version != sklearn.__version__:
        raise RuntimeError(f'Incompatibilidad: modelo scikit-learn {version}; runtime {sklearn.__version__}.')
    if artefacto.get('version') != '3.1':
        raise ValueError('El modelo no pertenece a Trayectoria V3.1.')
    return artefacto


def _validar_ids(df):
    """Valida identificadores sin convertirlos en variables del modelo."""
    if ID_COLUMNA not in df.columns:
        raise ValueError(f"Falta la columna obligatoria '{ID_COLUMNA}'.")
    ids = df[ID_COLUMNA]
    # Nunca convertir ID desde float para evitar perder ceros iniciales.
    if ids.isna().any():
        filas = (np.flatnonzero(ids.isna().to_numpy()) + 2)[:10].tolist()
        raise ValueError(f"Hay ID estudiante vacíos (filas CSV: {filas}).")
    ids = ids.astype('string').str.strip()
    vacios = ids.isna() | ids.eq('')
    if vacios.any():
        filas = (np.flatnonzero(vacios.to_numpy()) + 2)[:10].tolist()
        raise ValueError(f"Hay ID estudiante vacíos (filas CSV: {filas}).")
    if ids.duplicated().any():
        repetidos = ids[ids.duplicated(keep=False)].unique().tolist()
        raise ValueError(f'Hay ID estudiante repetidos dentro del archivo: {repetidos[:8]}')
    return ids


def validar_y_convertir(df_entrada, artefacto):
    """Defensa mínima del modelo; la validación principal se hace en la página."""
    if not isinstance(df_entrada, pd.DataFrame):
        raise TypeError('La entrada debe ser un DataFrame de pandas.')
    if df_entrada.empty:
        raise ValueError('El archivo no contiene estudiantes.')
    if df_entrada.columns.duplicated().any():
        raise ValueError('El archivo contiene nombres de columna duplicados.')
    _validar_ids(df_entrada)
    columnas = list(artefacto['columnas_pagina_a_uci'])
    faltantes = [col for col in columnas if col not in df_entrada.columns]
    if faltantes:
        raise ValueError(f'Faltan columnas obligatorias: {faltantes}')
    colisiones = [c for c in (*COLUMNAS_RESULTADO_ANTERIORES, *COLUMNAS_RESULTADO_NUEVAS) if c in df_entrada.columns]
    if colisiones:
        raise ValueError(f'El archivo ya contiene columnas calculadas: {colisiones}. Usa el original.')

    df = df_entrada[columnas].copy()
    for col in columnas:
        if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].astype('string').str.strip().replace('', pd.NA)

    faltantes = {
        col: (np.flatnonzero(df[col].isna().to_numpy()) + 2)[:8].tolist()
        for col in columnas if df[col].isna().any()
    }
    if faltantes:
        raise ValueError(f'Hay valores faltantes; filas CSV (encabezado=1): {faltantes}')

    for columna, mapa in artefacto['mapas_texto_a_codigo'].items():
        desconocidos = sorted(set(df[columna].astype(str).unique()) - set(mapa))
        if desconocidos:
            raise ValueError(f"Valores desconocidos en '{columna}': {desconocidos[:8]}")
        df[columna] = df[columna].map(mapa)
    df.rename(columns=artefacto['columnas_pagina_a_uci'], inplace=True)
    for col in df:
        try:
            df[col] = pd.to_numeric(df[col], errors='raise')
        except (ValueError, TypeError) as exc:
            raise ValueError(f"Valor numérico no válido en '{col}'.") from exc
    df = df[artefacto['features_modelo']]
    if df.isna().any().any() or not np.isfinite(df.to_numpy(dtype=float)).all():
        raise ValueError('Los datos contienen nulos o valores infinitos tras la conversión.')
    return df


def nivel_riesgo(probabilidad):
    """Tramos definidos por el proyecto (no probabilidades calibradas)."""
    return 'Bajo' if probabilidad < .30 else ('Medio' if probabilidad < .50 else 'Alto')


def _alertas_de_estudiante(fila):
    """Reglas orientativas explícitas. No establecen causas de deserción."""
    alertas = []
    recomendaciones = []

    def agregar(codigo, etiqueta, detalle, recomendacion):
        alertas.append({'codigo': codigo, 'etiqueta': etiqueta, 'detalle': detalle})
        if recomendacion not in recomendaciones:
            recomendaciones.append(recomendacion)

    ins1 = float(fila['Curricular units 1st sem (enrolled)'])
    ins2 = float(fila['Curricular units 2nd sem (enrolled)'])
    apr1 = float(fila['Curricular units 1st sem (approved)'])
    apr2 = float(fila['Curricular units 2nd sem (approved)'])
    nota1 = float(fila['Curricular units 1st sem (grade)'])
    nota2 = float(fila['Curricular units 2nd sem (grade)'])

    if ins1 > 0 and apr1 / ins1 < .50:
        agregar('baja_aprobacion_1s', 'Baja aprobación en primer semestre',
                f'{int(apr1)} de {int(ins1)} asignaturas inscritas aprobadas.',
                'Ofrecer tutorías y revisar obstáculos para aprobar asignaturas del primer semestre.')
    if ins2 > 0 and apr2 / ins2 < .50:
        agregar('baja_aprobacion_2s', 'Baja aprobación en segundo semestre',
                f'{int(apr2)} de {int(ins2)} asignaturas inscritas aprobadas.',
                'Revisar la progresión del segundo semestre y acordar un plan de apoyo académico.')
    # En el conjunto UCI portugués las calificaciones se expresan de 0 a 20;
    # un 0 con 0 aprobaciones podría ser ausencia de calificaciones y no una nota real.
    if 0 < nota1 < 10 and 0 < nota2 < 10:
        agregar('calificaciones_bajas', 'Calificaciones bajas en ambos semestres',
                f'Promedios registrados: {nota1:g} y {nota2:g} sobre escala 0–20.',
                'Ofrecer acompañamiento académico y revisar evaluaciones pendientes.')
    if int(fila['Tuition fees up to date']) == 0:
        agregar('arancel_pendiente', 'Aranceles no registrados al día',
                'El registro indica que el arancel no está al día.',
                'Ofrecer orientación sobre regularización y apoyos financieros disponibles.')
    if int(fila['Debtor']) == 1:
        agregar('deuda_registrada', 'Deuda registrada',
                'El registro indica la condición de deudor/a.',
                'Revisar con el área correspondiente opciones de apoyo financiero.')
    if int(fila['Displaced']) == 1:
        agregar('desplazamiento', 'Condición de desplazamiento registrada',
                'El estudiante figura como desplazado/a; esto no implica por sí solo una dificultad.',
                'Consultar, sin asumir problemas, si requiere apoyo de transporte o adaptación.')
    return alertas, recomendaciones


def enriquecer_dataframe(df_entrada, artefacto):
    datos_modelo = validar_y_convertir(df_entrada, artefacto)
    probabilidades = artefacto['modelo_riesgo'].predict_proba(datos_modelo)[:, 1]
    salida = df_entrada.copy().reset_index(drop=True)
    salida[ID_COLUMNA] = _validar_ids(df_entrada).reset_index(drop=True)
    salida['probabilidad_desercion'] = np.round(probabilidades, 6)
    salida['porcentaje_riesgo'] = np.round(probabilidades * 100, 2)
    salida['nivel_riesgo'] = [nivel_riesgo(p) for p in probabilidades]
    mascara = probabilidades >= artefacto['umbral_riesgo']
    salida['en_riesgo'] = np.where(mascara, 'Sí', 'No')
    salida['cluster'] = pd.Series(pd.NA, index=salida.index, dtype='Int64')
    salida['perfil'] = 'Sin perfil de riesgo'
    salida['factores_perfil'] = ''
    salida['intervencion_sugerida'] = ''
    if mascara.any():
        datos_cluster = datos_modelo.loc[mascara, artefacto['cols_cluster']]
        pred = artefacto['kmeans_perfiles'].predict(artefacto['scaler_cluster'].transform(datos_cluster))
        for pos, cluster in zip(np.flatnonzero(mascara), pred):
            perfil = artefacto['config_perfiles'][int(cluster)]
            salida.at[pos, 'cluster'] = int(cluster)
            for campo in ('perfil', 'factores_perfil', 'intervencion_sugerida'):
                salida.at[pos, campo] = perfil[campo]

    alertas = []
    recomendaciones = []
    for _, fila in datos_modelo.iterrows():
        avisos, sugerencias = _alertas_de_estudiante(fila)
        alertas.append(json.dumps(avisos, ensure_ascii=False))
        recomendaciones.append(json.dumps(sugerencias, ensure_ascii=False))
    salida['alertas_individuales'] = alertas
    salida['recomendaciones_individuales'] = recomendaciones
    return salida


def procesar_dataframe(df_entrada, ruta_artefacto=None):
    """Función que llama el backend cuando ya tiene un DataFrame validado."""
    return enriquecer_dataframe(df_entrada, cargar_artefacto(ruta_artefacto))


def procesar_csv(archivo_csv, ruta_artefacto=None, ruta_salida=None):
    """Recibe ruta o archivo binario. Conserva ceros iniciales de ID estudiante."""
    entrada = pd.read_csv(archivo_csv, sep=None, engine='python',
                          encoding='utf-8-sig', dtype={ID_COLUMNA: 'string'})
    salida = procesar_dataframe(entrada, ruta_artefacto)
    if ruta_salida is not None:
        salida.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
    return salida


def obtener_ficha_estudiante(resultado, id_estudiante):
    """Consulta un CSV/DataFrame YA enriquecido; no consulta datos personales externos.

    El control de acceso y aislamiento por lote/sesión corresponde al backend.
    """
    if ID_COLUMNA not in resultado.columns or not set(COLUMNAS_RESULTADO_NUEVAS).issubset(resultado.columns):
        raise ValueError('Se necesita un DataFrame enriquecido V3.1.')
    if id_estudiante is None or not str(id_estudiante).strip():
        raise ValueError('Se debe proporcionar un ID estudiante.')
    coincidencias = resultado.loc[resultado[ID_COLUMNA].astype(str) == str(id_estudiante).strip()]
    if len(coincidencias) != 1:
        raise ValueError('No existe un estudiante único con ese ID en el análisis actual.')
    fila = coincidencias.iloc[0]
    indicadores = {
        'edad_matricula': fila['Edad al momento de la matrícula'],
        'carrera': fila['Carrera'],
        'asignaturas_inscritas_1s': fila['Asignaturas del 1.er semestre (inscritas)'],
        'asignaturas_aprobadas_1s': fila['Asignaturas del 1.er semestre (aprobadas)'],
        'promedio_1s': fila['Promedio del 1.er semestre'],
        'asignaturas_inscritas_2s': fila['Asignaturas del 2.º semestre (inscritas)'],
        'asignaturas_aprobadas_2s': fila['Asignaturas del 2.º semestre (aprobadas)'],
        'promedio_2s': fila['Promedio del 2.º semestre'],
        'aranceles_al_dia': fila['Aranceles al día'],
        'deudor': fila['Deudor/a'],
        'desplazado': fila['Desplazado/a'],
    }
    return {
        'id_estudiante': str(fila[ID_COLUMNA]),
        'porcentaje_riesgo': float(fila['porcentaje_riesgo']),
        'nivel_riesgo': str(fila['nivel_riesgo']),
        'en_riesgo': str(fila['en_riesgo']),
        'cluster': None if pd.isna(fila['cluster']) else int(fila['cluster']),
        'perfil': str(fila['perfil']),
        'factores_perfil': str(fila['factores_perfil']),
        'intervencion_sugerida': str(fila['intervencion_sugerida']),
        'alertas_individuales': json.loads(fila['alertas_individuales']),
        'recomendaciones_individuales': json.loads(fila['recomendaciones_individuales']),
        'indicadores': {k: (v.item() if isinstance(v, np.generic) else v) for k, v in indicadores.items()},
    }


In [ ]:
from trayectoria_inferencia import procesar_dataframe, procesar_csv, obtener_ficha_estudiante, validar_y_convertir
print("Inferencia V3.1 lista: CSV/DataFrame + búsqueda de ficha por ID.")

## 7. Exportar **modelo nuevo** y manifiesto

Se guarda con los mismos nombres de archivo que la integración anterior, pero **este es un modelo nuevo reentrenado con 1.9.1**. No basta con editar el manifest de un pickle antiguo. Se genera una suma SHA-256 que el módulo de inferencia valida antes de cargar.


In [ ]:
from hashlib import sha256

METADATOS = {
    "scikit_learn": sklearn.__version__,
    "imbalanced_learn": importlib.metadata.version('imbalanced-learn'),
    "narwhals": importlib.metadata.version('narwhals'),
    "version_trayectoria": "3.1",
    "migracion_sklearn": "1.8.0 -> 1.9.1 (REENTRENADO)",
    "modelo_riesgo_reentrenado": True,
    "kmeans_reentrenado": True,
    "se_requiere_id_estudiante": True,
    "reglas_alertas": "v3.1-prototipo-Portugal-0a20",
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "pandas": pd.__version__,
    "python": platform.python_version(),
    "origen": "data.csv UCI Portugal ID 697",
    "n_entrenamiento": len(X_train),
    "n_test": len(X_test),
    "resultado_f1": float(resultados["Random Forest"]["F1_desercion"]),
    "resultado_roc_auc": float(resultados["Random Forest"]["ROC_AUC"]),
    "silhouette": {str(k): float(v) for k, v in silhouettes.items()},
}
artefacto_produccion = {
    "version": "3.1",
    "modelo_riesgo": modelo_riesgo,
    "kmeans_perfiles": kmeans_perfiles,
    "scaler_cluster": scaler_cluster,
    "features_modelo": FEATURES_MODELO,
    "cols_cluster": COLS_CLUSTER,
    "umbral_riesgo": UMBRAL_RIESGO,
    "columnas_pagina_a_uci": COLUMNAS_PAGINA_A_UCI,
    "mapas_texto_a_codigo": MAPAS_TEXTO_A_CODIGO,
    "config_perfiles": CONFIG_PERFILES,
    "metadatos": METADATOS,
}
NOMBRE_MODELO = "trayectoria_pipeline_v3_1.pkl"
joblib.dump(artefacto_produccion, NOMBRE_MODELO)
hash_archivo = sha256(Path(NOMBRE_MODELO).read_bytes()).hexdigest()
MANIFEST_EXPORTADO = {**METADATOS, "archivo_sha256": hash_archivo}
Path("trayectoria_pipeline_v3_1.manifest.json").write_text(
    json.dumps(MANIFEST_EXPORTADO, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Modelo REAL 1.9.1 y manifest SHA-256 generados:", NOMBRE_MODELO)


In [ ]:
Path("requirements_inferencia.txt").write_text('# Entorno de inferencia para el nuevo .pkl generado con scikit-learn 1.9.1.\n# Python >=3.11\nscikit-learn==1.9.1\nnumpy==2.3.5\nscipy==1.17.0\npandas==2.2.3\njoblib==1.5.3\nthreadpoolctl==3.6.0\nnarwhals==2.21.2\n',encoding="utf-8")
Path("requirements_entrenamiento.txt").write_text('-r requirements_inferencia.txt\n# 0.14.2 añade compatibilidad con scikit-learn 1.9.x.\nimbalanced-learn==0.14.2\n# Solo si el dataset local data.csv no existe:\nucimlrepo>=0.0.7,<1.0\n# XLSX opcional para re-exportar plantillas; no es parte del modelo\nopenpyxl==3.1.5\n',encoding="utf-8")
print("Dependencias para página y entrenamiento actualizadas a 1.9.1")

## 8. Generar datos de prueba con IDs y verificar compatibilidad

Si no se ha subido el CSV portugués con IDs, este cuaderno lo reconstruye desde el dataset UCI original usando **exactamente** los nombres y categorías de la plantilla española. Se comprueba la equivalencia de sus 36 variables con las originales.


In [ ]:
def crear_dataset_demo_espanol(X_uci):
    columnas = {}
    for columna_pagina, columna_uci in COLUMNAS_PAGINA_A_UCI.items():
        valores = X_uci[columna_uci].reset_index(drop=True)
        if columna_pagina in MAPAS_TEXTO_A_CODIGO:
            mapa_inverso = {codigo: texto for texto, codigo in MAPAS_TEXTO_A_CODIGO[columna_pagina].items()}
            nuevos = valores.map(mapa_inverso)
            if nuevos.isna().any():
                raise ValueError(f"La columna {columna_pagina} contiene categorías UCI sin equivalencia española.")
            columnas[columna_pagina] = nuevos
        else:
            columnas[columna_pagina] = valores
    df = pd.DataFrame(columnas)
    df.insert(0, 'ID estudiante', [f'EST-{i:06d}' for i in range(1, len(df) + 1)])
    return df

# Siempre generar el ejemplo a partir de la misma fuente usada para entrenamiento.
demo_entrada = crear_dataset_demo_espanol(X)
demo_entrada.to_csv('dataset_portugal_con_ID_v3_1.csv', index=False, encoding='utf-8-sig')

# La conversión español -> numérico UCI debe ser exacta.
X_reconstruido = validar_y_convertir(demo_entrada, artefacto_produccion)
pd.testing.assert_frame_equal(
    X_reconstruido.reset_index(drop=True).astype(float),
    X.reset_index(drop=True).astype(float),
    check_dtype=False, check_exact=True
)
print('[OK] Plantilla española (37 columnas) convertida exactamente a las 36 variables UCI.')

demo_salida = procesar_dataframe(demo_entrada, NOMBRE_MODELO)
demo_salida.to_csv('dataset_portugal_enriquecido_v3_1.csv', index=False, encoding='utf-8-sig')
assert len(demo_salida) == len(demo_entrada)
assert demo_salida.columns[:37].tolist() == demo_entrada.columns.tolist()
assert demo_salida['ID estudiante'].is_unique
print('Estudiantes procesados:',len(demo_salida),'Columnas:',len(demo_salida.columns))
display(demo_salida[['ID estudiante','porcentaje_riesgo','nivel_riesgo','perfil','alertas_individuales']].head())
ficha_demo = obtener_ficha_estudiante(demo_salida, str(demo_salida.iloc[0]['ID estudiante']))
print('Ficha individual:', json.dumps(ficha_demo,ensure_ascii=False,indent=2))


## 9. Validación automatizada y ZIP listo para integración

Se ejecuta la comparación con la versión anterior **si existe** el CSV de referencia. La comparación **no obliga** a que las probabilidades sean idénticas entre dos versiones distintas. El reporte indica cuánto variaron.

El ZIP resultante se entrega a tus compañeros para sustituir juntos `.pkl`, `.manifest.json`, `.py` y `requirements_inferencia.txt` (primero en un entorno de prueba).


In [ ]:
%%writefile verificar_migracion.py
"""Pruebas de integración para Trayectoria V3.1 con scikit-learn 1.9.1.

Ejecutar DESPUÉS de entrenar el nuevo .pkl, desde la misma carpeta.
Este script comprueba archivos, incompatibilidades, dataset y ficha por ID.
"""
from pathlib import Path
import json
import importlib.metadata

import numpy as np
import pandas as pd
import sklearn

from trayectoria_inferencia import (
    procesar_dataframe, obtener_ficha_estudiante, cargar_artefacto,
    COLUMNAS_RESULTADO_ANTERIORES, COLUMNAS_RESULTADO_NUEVAS,
)

BASE = Path(__file__).resolve().parent


def _debe_fallar(df, texto):
    try:
        procesar_dataframe(df)
    except (ValueError, TypeError) as exc:
        print(f'  [OK] Rechazo correcto de {texto}: {str(exc)[:115]}')
    else:
        raise AssertionError(f'La validación no rechazó {texto}')


def ejecutar_verificacion(ruta_entrada=None, ruta_salida=None):
    if sklearn.__version__ != '1.9.1':
        raise RuntimeError('Para verificar este paquete se necesita scikit-learn 1.9.1, no ' + sklearn.__version__)
    artefacto = cargar_artefacto()
    assert artefacto['metadatos']['scikit_learn'] == '1.9.1'
    entrada_path = Path(ruta_entrada) if ruta_entrada else BASE / 'dataset_portugal_con_ID_v3_1.csv'
    if not entrada_path.exists():
        raise FileNotFoundError(f'Falta el dataset de demostración: {entrada_path}')
    entrada = pd.read_csv(entrada_path, encoding='utf-8-sig', dtype={'ID estudiante':'string'})
    assert entrada.shape[1] == 37, f'El CSV debería tener 37 columnas: {entrada.shape}'
    assert entrada['ID estudiante'].is_unique
    salida = procesar_dataframe(entrada)
    assert len(salida) == len(entrada)
    assert salida.columns[:37].tolist() == entrada.columns.tolist()
    assert set(COLUMNAS_RESULTADO_ANTERIORES + COLUMNAS_RESULTADO_NUEVAS).issubset(salida.columns)
    assert salida['probabilidad_desercion'].between(0, 1).all()
    assert salida['porcentaje_riesgo'].between(0, 100).all()
    assert (salida.loc[salida['en_riesgo'] == 'No', 'cluster'].isna()).all()
    assert (salida.loc[salida['en_riesgo'] == 'Sí', 'cluster'].notna()).all()
    ids = salida['ID estudiante']
    ejemplo_id = str(ids.iloc[0])
    ficha = obtener_ficha_estudiante(salida, ejemplo_id)
    assert ficha['id_estudiante'] == ejemplo_id
    assert isinstance(ficha['alertas_individuales'], list)
    assert isinstance(ficha['recomendaciones_individuales'], list)
    assert isinstance(ficha['porcentaje_riesgo'], float)
    assert isinstance(ficha['indicadores'], dict)
    print(f'[OK] Ficha individual: {ejemplo_id}')

    # Validaciones en memoria: no generan archivos incorrectos de prueba.
    repetido = entrada.iloc[:2].copy().reset_index(drop=True)
    repetido.loc[1,'ID estudiante'] = repetido.loc[0,'ID estudiante']
    _debe_fallar(repetido, 'ID duplicado')
    incompleto = entrada.iloc[:1].copy().reset_index(drop=True)
    incompleto.loc[0,'ID estudiante'] = pd.NA
    _debe_fallar(incompleto, 'ID faltante')
    invalido = entrada.iloc[:1].copy().reset_index(drop=True)
    invalido.loc[0,'Carrera'] = 'CARRERA_DESCONOCIDA'
    _debe_fallar(invalido, 'categoría desconocida')

    salida_path = Path(ruta_salida) if ruta_salida else BASE / 'dataset_portugal_enriquecido_v3_1.csv'
    salida.to_csv(salida_path, index=False, encoding='utf-8-sig')
    ficha_path = BASE / 'ejemplo_ficha_estudiante.json'
    ficha_path.write_text(json.dumps(ficha, ensure_ascii=False, indent=2), encoding='utf-8')
    reporte = {
        'sklearn': sklearn.__version__,
        'imbalanced_learn': importlib.metadata.version('imbalanced-learn'),
        'total_estudiantes': len(salida),
        'estudiantes_en_riesgo': int(salida['en_riesgo'].eq('Sí').sum()),
        'columnas_entrada': entrada.shape[1],
        'columnas_salida': salida.shape[1],
        'ficha_ejemplo': ejemplo_id,
        'comparacion_v3_1_1_8_0': {'estado':'no disponible'},
    }
    ref = BASE / 'referencia_predicciones_v3_1_sklearn_1_8_0.csv'
    if ref.exists():
        anterior = pd.read_csv(ref, encoding='utf-8-sig', dtype={'ID estudiante':'string'})
        comparado = salida[['ID estudiante', 'probabilidad_desercion','en_riesgo','cluster']].merge(
            anterior[['ID estudiante', 'probabilidad_desercion','en_riesgo','cluster']],
            on='ID estudiante', how='outer', validate='one_to_one', suffixes=('_nueva','_anterior'), indicator=True
        )
        assert comparado['_merge'].eq('both').all(), 'Cambió la cantidad de IDs o la correspondencia entre registros'
        delta = (comparado['probabilidad_desercion_nueva'] - comparado['probabilidad_desercion_anterior']).abs()
        old_cluster = comparado['cluster_anterior'].fillna(-1).astype(int)
        new_cluster = comparado['cluster_nueva'].fillna(-1).astype(int)
        cambios_nivel = int((comparado['en_riesgo_nueva'] != comparado['en_riesgo_anterior']).sum())
        cambios_cluster = int((old_cluster != new_cluster).sum())
        reporte['comparacion_v3_1_1_8_0'] = {
            'estado':'comparado',
            'cambio_probabilidad_medio_absoluto': float(delta.mean()),
            'cambio_probabilidad_maximo_absoluto': float(delta.max()),
            'estudiantes_que_cambiaron_riesgo_binario': cambios_nivel,
            'estudiantes_que_cambiaron_cluster': cambios_cluster,
            'interpretacion': 'Es posible que una actualización de biblioteca cambie valores. Revisar antes de reemplazar el modelo en producción.'
        }
        print('[OK] Comparación contra scikit-learn 1.8.0:')
        print(json.dumps(reporte['comparacion_v3_1_1_8_0'], ensure_ascii=False, indent=2))
        # No hay identidad garantizada entre dos modelos ENTRENADOS por versiones distintas.
        # Se informa para revisión humana sin ocultar cambios.
    (BASE / 'reporte_verificacion_1_9_1.json').write_text(
        json.dumps(reporte, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    print(f'[OK] {len(salida)} estudiantes, {salida.shape[1]} columnas, {reporte["estudiantes_en_riesgo"]} en riesgo')
    print(f'[OK] Nuevo modelo realmente entrenado con scikit-learn {sklearn.__version__}')
    return reporte


if __name__ == '__main__':
    ejecutar_verificacion()


In [ ]:
from verificar_migracion import ejecutar_verificacion
reporte = ejecutar_verificacion()
print('Verificación terminada. Resumen:')
print(json.dumps(reporte,ensure_ascii=False,indent=2))

In [ ]:
# Plantilla de encabezados española para subir a la página.
# Si ya se aportó una plantilla .xlsx conservaremos esa; de lo contrario
# se crea una equivalente con las 37 cabeceras, sin información personal.
if not Path('Plantilla_Trayectoria_V3_1.xlsx').exists():
    demo_entrada.head(0).to_excel('Plantilla_Trayectoria_V3_1.xlsx',index=False)
print('Plantilla disponible:',Path('Plantilla_Trayectoria_V3_1.xlsx').is_file())

In [ ]:
# Empaquetado para tus compañeros; se crea solo tras superar todas las pruebas.
from zipfile import ZipFile, ZIP_DEFLATED
from pathlib import Path

INSTRUCCIONES = """# Trayectoria IA V3.1 — scikit-learn 1.9.1 (REENTRENADO)

Este ZIP es el paquete generado mediante el cuaderno y contiene un modelo
realmente entrenado con 1.9.1; el manifest registra la versión y SHA-256.

## Instalación en el BACKEND (Python >=3.11)
1. Crear o activar un entorno virtual para el servidor.
2. pip install -r requirements_inferencia.txt
3. python verificar_migracion.py  # solo si se adjunta el archivo de verificación

## Integración
from trayectoria_inferencia import procesar_dataframe, obtener_ficha_estudiante
resultado = procesar_dataframe(df_validado)
ficha = obtener_ficha_estudiante(resultado, 'EST-000001')

La entrada tiene 37 columnas: ID estudiante y las 36 variables previas.
El módulo devuelve 47 columnas, incluyendo riesgo, cluster, perfil y alertas.

Reemplazar JUNTOS el .pkl, el .manifest.json, trayectoria_inferencia.py y requirements.
No mezclar este .pkl nuevo con bibliotecas 1.8.0 ni editar un manifest viejo para fingir migración.
El ID se conserva, no se usa como variable predictora.

Las probabilidades son estimaciones de un modelo entrenado en estudiantes
portugueses; requieren validación local antes de decisiones reales.
"""
Path('LEEME_ENTREGA_1_9_1.md').write_text(INSTRUCCIONES,encoding='utf-8')

archivos_web = [
    'trayectoria_pipeline_v3_1.pkl',
    'trayectoria_pipeline_v3_1.manifest.json',
    'trayectoria_inferencia.py',
    'requirements_inferencia.txt',
    'requirements_entrenamiento.txt',
    'verificar_migracion.py',
    'Plantilla_Trayectoria_V3_1.xlsx',
    'dataset_portugal_con_ID_v3_1.csv',
    'dataset_portugal_enriquecido_v3_1.csv',
    'ejemplo_ficha_estudiante.json',
    'reporte_verificacion_1_9_1.json',
    'LEEME_ENTREGA_1_9_1.md',
]
zip_final=Path('Trayectoria_V3_1_LISTO_PARA_WEB_sklearn_1_9_1.zip')
with ZipFile(zip_final,'w',compression=ZIP_DEFLATED,compresslevel=6) as zf:
    for nombre in archivos_web:
        p=Path(nombre)
        if not p.is_file():
            raise FileNotFoundError(f'Falta un archivo esencial para la entrega: {p}')
        zf.write(p,p.name)
print('[OK] Paquete de inferencia 1.9.1 generado:',zip_final,zip_final.stat().st_size,'bytes')

try:
    from google.colab import files
    files.download(str(zip_final))
except ImportError:
    print('Fuera de Google Colab: descarga manualmente el ZIP recién generado.')